In [13]:
import sys, argparse
import numpy as np
from pathlib import Path
import cv2
import torch
import plotly.graph_objects as go
from tqdm.auto import tqdm

DROID_ROOT = Path('/users/tnanni/ghost/DROID-SLAM')
sys.path.insert(0, str(DROID_ROOT))
sys.path.insert(0, str(DROID_ROOT / 'droid_slam'))

DROID_WEIGHTS  = '/capstor/scratch/cscs/tnanni/ghost_checkpoints/droid.pth'
RICH_DATA_ROOT = Path('/tmp/rich_mount')
SCENE          = 'Pavallion_002_plankjack'
CAM            = 'cam_10'

_FX, _FY = 1200.0, 1200.0
_CX, _CY =  720.0,  526.0

In [14]:
# ── Run DROID-SLAM ──────────────────────────────────────────────────────────────

def slam_image_stream(frames_dir, fx, fy, cx, cy):
    img_files = sorted(frames_dir.glob('*.jpeg')) or sorted(frames_dir.glob('*.jpg'))
    for t, imfile in enumerate(img_files):
        image = cv2.imread(str(imfile))
        h0, w0 = image.shape[:2]
        scale  = np.sqrt((384 * 512) / (h0 * w0))
        h1, w1 = int(h0 * scale), int(w0 * scale)
        image  = cv2.resize(image, (w1, h1))
        image  = image[:h1 - h1 % 8, :w1 - w1 % 8]
        image  = torch.as_tensor(image).permute(2, 0, 1)
        intr   = torch.tensor([fx * w1/w0, fy * h1/h0, cx * w1/w0, cy * h1/h0])
        yield t, image[None], intr


def run_droid_slam(frames_dir, fx, fy, cx, cy, weights):
    from droid import Droid
    torch.multiprocessing.set_start_method('spawn', force=True)

    args = argparse.Namespace(
        weights=weights, image_size=[240, 320], buffer=512,
        stereo=False, disable_vis=True, beta=0.3, filter_thresh=2.4,
        warmup=8, keyframe_thresh=4.0, frontend_thresh=16.0,
        frontend_window=25, frontend_radius=2, frontend_nms=1,
        backend_thresh=22.0, backend_radius=2, backend_nms=3, upsample=False,
    )

    stream = list(slam_image_stream(frames_dir, fx, fy, cx, cy))
    droid  = None
    for t, image, intrinsics in tqdm(stream, desc='DROID-SLAM'):
        if droid is None:
            H, W = image.shape[2], image.shape[3]
            args.image_size = [H, W]
            droid = Droid(args)
        droid.track(t, image, intrinsics=intrinsics)

    poses  = droid.terminate(iter(stream))          # (N, 7) T_wc
    n_kf   = droid.video.counter.value
    disps  = droid.video.disps[:n_kf].cpu().numpy()
    tstamp = droid.video.tstamp[:n_kf].cpu().numpy().astype(int)
    del droid; torch.cuda.empty_cache()
    return poses, disps, tstamp


frames_dir = RICH_DATA_ROOT / SCENE / CAM
poses, disps, tstamp = run_droid_slam(frames_dir, _FX, _FY, _CX, _CY, DROID_WEIGHTS)
print(f'poses: {poses.shape}  keyframes: {len(tstamp)}  tstamp range: {tstamp[0]}–{tstamp[-1]}')
print(f'camera position range (unscaled):  '
      f'x={poses[:,0].min():.3f}–{poses[:,0].max():.3f}  '
      f'y={poses[:,1].min():.3f}–{poses[:,1].max():.3f}  '
      f'z={poses[:,2].min():.3f}–{poses[:,2].max():.3f}')

DROID-SLAM:   0%|          | 1/590 [00:00<01:45,  5.56it/s]

/capstor/scratch/cscs/tnanni/ghost_checkpoints/droid.pth


DROID-SLAM: 100%|██████████| 590/590 [00:06<00:00, 97.17it/s] 


################################
################################
poses: (590, 7)  keyframes: 16  tstamp range: 0–584
camera position range (unscaled):  x=-0.439–0.218  y=-0.336–0.047  z=-0.135–0.092


In [15]:
# ── Load body data to get scale λ and world-frame people positions ──────────────

BASE = Path(f'/iopsstor/scratch/cscs/tnanni/ghost_outputs/rich11_segmentation_test/{SCENE}')

def load_tracks(cam):
    persons = {}
    for npz in sorted((BASE / cam / 'body_data').glob('person_*.npz')):
        pid = int(npz.stem.split('_')[1])
        with np.load(str(npz)) as d:
            if 'pred_keypoints_3d' not in d: continue
            entry = {'kpts3d': d['pred_keypoints_3d'].copy(), 'frames': d['frame_indices'].copy()}
            for key in ('pred_cam_t', 'smplx_transl'):
                if key in d:
                    entry['pred_cam_t'] = d[key].copy(); break
            if 'smplx_transl' in d:
                entry['smplx_transl'] = d['smplx_transl'].copy()
            persons[pid] = entry
    return persons

tracks = load_tracks(CAM)
frame_offset = int(min(d['frames'].min() for d in tracks.values()))
print(f'frame_offset={frame_offset}, persons={sorted(tracks.keys())}')
for pid, data in tracks.items():
    has_smplx = 'smplx_transl' in data
    print(f'  P{pid}: pred_cam_t={data.get("pred_cam_t") is not None}  smplx_transl={has_smplx}')

# Scale estimation
H_orig, W_orig = cv2.imread(str(next(frames_dir.glob('*.jpeg')))).shape[:2]
H_disp, W_disp = disps.shape[1], disps.shape[2]
sx, sy = W_disp / W_orig, H_disp / H_orig

lambdas = []
for pid, data in tracks.items():
    cam_t = data.get('pred_cam_t')
    if cam_t is None: continue
    f2i = {int(f): i for i, f in enumerate(data['frames'])}
    for kf_i, t in enumerate(tstamp):
        arr_idx = f2i.get(int(t) + frame_offset)
        if arr_idx is None: continue
        x, y, z = cam_t[arr_idx]
        if z < 0.5: continue
        px = int(round((_FX * x / z + _CX) * sx))
        py = int(round((_FY * y / z + _CY) * sy))
        if not (0 <= px < W_disp and 0 <= py < H_disp): continue
        d_inv = float(disps[kf_i, py, px])
        if d_inv < 1e-6: continue
        lambdas.append(z * d_inv)

lam = float(np.median(lambdas)) if lambdas else 1.0
print(f'λ = {lam:.4f}  ({len(lambdas)} detections)')

frame_offset=110, persons=[1, 2]
  P1: pred_cam_t=True  smplx_transl=True
  P2: pred_cam_t=True  smplx_transl=True
λ = 4.8431  (27 detections)


In [16]:
# ── Apply transform: cam-local → world frame ────────────────────────────────────

def quat_to_rot(q):
    qx, qy, qz, qw = q
    return np.array([
        [1-2*(qy**2+qz**2),  2*(qx*qy-qz*qw),  2*(qx*qz+qy*qw)],
        [2*(qx*qy+qz*qw),  1-2*(qx**2+qz**2),  2*(qy*qz-qx*qw)],
        [2*(qx*qz-qy*qw),    2*(qy*qz+qx*qw),  1-2*(qx**2+qy**2)],
    ])

def cam_to_world(root_cam_seq, frames, poses, frame_offset, lam):
    """Apply SLAM correction to a (T,3) camera-frame root sequence."""
    out = np.full((len(frames), 3), np.nan)
    for i, frame_idx in enumerate(frames):
        t = int(frame_idx) - frame_offset
        if t < 0 or t >= len(poses): continue
        R = quat_to_rot(poses[t, 3:])
        out[i] = R @ root_cam_seq[i] + lam * poses[t, :3]
    return out

# pred_cam_t → world  (primary source)
world_from_pred = {}
for pid, data in tracks.items():
    if data.get('pred_cam_t') is None: continue
    world_from_pred[pid] = cam_to_world(data['pred_cam_t'], data['frames'], poses, frame_offset, lam)
    valid = np.isfinite(world_from_pred[pid][:, 0]).sum()
    print(f'P{pid} pred_cam_t: {valid}/{len(data["frames"])} valid frames')

# smplx_transl → world  (for comparison)
world_from_smplx = {}
for pid, data in tracks.items():
    if 'smplx_transl' not in data: continue
    world_from_smplx[pid] = cam_to_world(data['smplx_transl'], data['frames'], poses, frame_offset, lam)
    valid = np.isfinite(world_from_smplx[pid][:, 0]).sum()
    print(f'P{pid} smplx_transl: {valid}/{len(data["frames"])} valid frames')

P1 pred_cam_t: 558/558 valid frames
P2 pred_cam_t: 590/590 valid frames
P1 smplx_transl: 558/558 valid frames
P2 smplx_transl: 590/590 valid frames


In [17]:
# ── Visualize ───────────────────────────────────────────────────────────────────

cam_pos = lam * poses[:, :3]   # (N, 3) metric camera positions
t_axis  = np.arange(len(cam_pos))

fig = go.Figure()

# camera trajectory
fig.add_trace(go.Scatter3d(
    x=cam_pos[:, 0], y=cam_pos[:, 1], z=cam_pos[:, 2],
    mode='lines+markers',
    marker=dict(size=2, color=t_axis, colorscale='Viridis', showscale=True,
                colorbar=dict(title='frame', x=1.0)),
    line=dict(color='grey', width=1),
    name='cam_10 trajectory'
))

# keyframes
kf_pos = lam * poses[tstamp, :3]
fig.add_trace(go.Scatter3d(
    x=kf_pos[:, 0], y=kf_pos[:, 1], z=kf_pos[:, 2],
    mode='markers', marker=dict(size=6, color='red', symbol='diamond'),
    name='keyframes'
))

palette = {1: ('blue', 'cyan'), 2: ('orange', 'gold'), 3: ('green', 'lime'), 4: ('purple', 'violet')}

for pid in sorted(world_from_pred.keys()):
    col_pred, col_smplx = palette.get(pid, ('black', 'grey'))
    # pred_cam_t trajectory (solid)
    pos = world_from_pred[pid]
    valid = np.isfinite(pos[:, 0])
    fig.add_trace(go.Scatter3d(
        x=pos[valid, 0], y=pos[valid, 1], z=pos[valid, 2],
        mode='lines', line=dict(color=col_pred, width=4),
        name=f'P{pid} pred_cam_t'
    ))
    # smplx_transl trajectory (dashed, same person)
    if pid in world_from_smplx:
        pos2 = world_from_smplx[pid]
        valid2 = np.isfinite(pos2[:, 0])
        fig.add_trace(go.Scatter3d(
            x=pos2[valid2, 0], y=pos2[valid2, 1], z=pos2[valid2, 2],
            mode='lines', line=dict(color=col_smplx, width=2, dash='dash'),
            name=f'P{pid} smplx_transl'
        ))

fig.update_layout(
    title=f'DROID-SLAM: {CAM} — {SCENE} (λ={lam:.3f})',
    scene=dict(aspectmode='data', xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Z (m)'),
    height=700
)
fig.show()